In [ ]:
# from datasets import load_dataset
# ds = load_dataset("Bose345/sp500_earnings_transcripts") # original dataset: kurry/sp500_earnings_transcripts
# data = ds["train"]
# df = data.to_pandas()
# df.to_csv('earnings_transcripts.csv', index=False)

In [1]:
import pandas as pd
from datetime import timedelta
import yfinance as yf

In [2]:
df = pd.read_csv("earnings_transcripts.csv")

In [3]:
df.head()

,symbol,quarter,year,date,content,structured_content,company_name,company_id
0,A,4,2020,2020-11-23 16:30:00,"Operator: Good afternoon, and welcome to the A...","[{'speaker': 'Operator', 'text': ""Good afterno...","Agilent Technologies, Inc.",154924.0
1,A,3,2020,2020-08-18 16:30:00,"Operator: Good afternoon, and welcome to the A...","[{'speaker': 'Operator', 'text': ""Good afterno...","Agilent Technologies, Inc.",154924.0
2,A,2,2020,2020-05-21 16:30:00,"Operator: Good afternoon, and welcome to the A...","[{'speaker': 'Operator', 'text': ""Good afterno...","Agilent Technologies, Inc.",154924.0
3,A,1,2020,2020-02-18 16:30:00,Operator: Good afternoon and welcome to the Ag...,"[{'speaker': 'Operator', 'text': ""Good afterno...","Agilent Technologies, Inc.",154924.0
4,A,4,2021,2021-11-22 16:30:00,Operator: Good afternoon and welcome to the Ag...,"[{'speaker': 'Operator', 'text': ""Good afterno...","Agilent Technologies, Inc.",154924.0


In [7]:
print(df["date"].dtype)

str


In [9]:
df["date"] = pd.to_datetime(df["date"])
df = df.reset_index(drop=True)
df["event_id"] = df.index

print(df[["event_id", "symbol", "date"]].head())
print(df["date"].dtype)
print(df.columns)

   event_id symbol                date
0         0      A 2020-11-23 16:30:00
1         1      A 2020-08-18 16:30:00
2         2      A 2020-05-21 16:30:00
3         3      A 2020-02-18 16:30:00
4         4      A 2021-11-22 16:30:00
datetime64[us]
Index(['symbol', 'quarter', 'year', 'date', 'content', 'structured_content',
       'company_name', 'company_id', 'event_id'],
      dtype='str')


In [13]:
def get_stock_data(
    symbol,
    start_date,
    end_date
):
    
    try:
        stock = yf.download(
            symbol,
            start=start_date,
            end=end_date,
            interval="1d",
            auto_adjust=True,
            progress=False,
            threads=False
        )
    except Exception:
        return None

    # In case yfinance has no data
    if stock.empty:
        return None

    # Handle yfinance MultiIndex columns
    if isinstance(stock.columns, pd.MultiIndex):
        stock.columns = stock.columns.get_level_values(0)
    # Remove rows without a closing price
    stock = stock.dropna(subset=["Close"])
    if stock.empty:
        return None

    # Normalize and sort dates once
    stock.index = pd.to_datetime(stock.index).normalize()
    stock = stock.sort_index()

    return stock

In [14]:
# Store downloaded market data
market_data = {}

# Store tickers for which yfinance failed
failed_symbols = []

for symbol, group in df.groupby("symbol"):

    # Earliest and latest earnings dates for this company
    min_date = group["date"].min()
    max_date = group["date"].max()
    start_date = min_date - pd.Timedelta(days=60)
    end_date = max_date + pd.Timedelta(days=15)

    stock = get_stock_data(
        symbol,
        start_date,
        end_date
    )

    if stock is None:
        failed_symbols.append(symbol)
        continue

    market_data[symbol] = stock

$ABMD: possibly delisted; no timezone found

1 Failed download:
['ABMD']: possibly delisted; no timezone found
$ADCT: Data doesn't exist for startDate = 1136152800, endDate = 1198879200

1 Failed download:
['ADCT']: Data doesn't exist for startDate = 1136152800, endDate = 1198879200
$ADS: possibly delisted; no timezone found

1 Failed download:
['ADS']: possibly delisted; no timezone found
$ADT: Data doesn't exist for startDate = 1362231000, endDate = 1455715800

1 Failed download:
['ADT']: Data doesn't exist for startDate = 1362231000, endDate = 1455715800
$AKS: possibly delisted; no timezone found

1 Failed download:
['AKS']: possibly delisted; no timezone found
$ALTR: possibly delisted; no timezone found

1 Failed download:
['ALTR']: possibly delisted; no timezone found
$ALXN: possibly delisted; no timezone found

1 Failed download:
['ALXN']: possibly delisted; no timezone found
$AMBC: possibly delisted; no timezone found

1 Failed download:
['AMBC']: possibly delisted; no timezone 

In [24]:
len(failed_symbols), len(market_data), df.groupby("symbol").size().shape[0]

(102, 583, 685)

In [16]:
market_data["AAPL"]

Price,Close,High,Low,Open,Volume
Date,,,,,
2005-08-15,1.426091,1.445532,1.389303,1.390199,1086727600
2005-08-16,1.383321,1.420708,1.382124,1.417417,537622400
2005-08-17,1.410240,1.418913,1.386909,1.387807,499724400
2005-08-18,1.384816,1.405753,1.368367,1.403061,442559600
2005-08-19,1.370758,1.396779,1.368964,1.384217,376569200
...,...,...,...,...,...
2025-05-12,209.776123,210.253825,205.755562,209.955265,63775800
2025-05-13,211.905838,212.373579,207.994748,209.417863,51909300
2025-05-14,211.308716,212.910972,209.567133,211.408226,49325800


In [17]:
def build_event_window(
    stock,
    date,
    pre_days=30,
    post_days=7
):

    stock = stock.copy()

    # Normalize the index column of stock data from yfinance, i.e. market-data dates
    stock.index = pd.to_datetime(stock.index).normalize()
    stock = stock.sort_index()

    event_timestamp = pd.Timestamp(date)
    event_date = event_timestamp.normalize()

    # Compare market closing time to the event time on the event day
    market_close = event_date + pd.Timedelta(hours=16)
    after_close = event_timestamp >= market_close


    # Before-event
    if after_close:
        # The closing price on event_date is considered as pre-event data
        pre = stock[stock.index <= event_date].tail(pre_days)

    else:
        # The closing price on event_date is NOT pre-event information
        pre = stock[stock.index < event_date].tail(pre_days)

    # After-event
    if after_close:
        # First reaction is next trading day
        post = stock[stock.index > event_date].head(post_days)

    else:
        # Event-day close captures the initial reaction
        post = stock[stock.index >= event_date].head(post_days + 1)

    if len(pre) < pre_days:
        return None

    if after_close:
        if len(post) < post_days:
            return None
    else:
        if len(post) < post_days + 1:
            return None

    # Relative trading-day labels
    if after_close:
        pre["relative_day"] = range(-(pre_days - 1), 1)
        post["relative_day"] = range(1, post_days + 1)

    else:
        pre["relative_day"] = range(-pre_days, 0)
        post["relative_day"] = range(0, post_days + 1)

    pre["window"] = "pre"
    post["window"] = "post"

    return pd.concat([pre, post])

In [20]:
market_events = []
valid_events = []
failed_events = []

for _, event in df.iterrows():

    symbol = event["symbol"]
    date = event["date"]

    # Get already-downloaded market data
    stock = market_data.get(symbol)

    # Yahoo had no data for this ticker
    if stock is None:
        failed_events.append(event["event_id"])
        continue

    # Build the event window locally
    window = build_event_window(
        stock,
        date
    )

    # Not enough trading history
    if window is None:
        failed_events.append(event["event_id"])
        continue

    window["event_id"] = event["event_id"]
    window["symbol"] = symbol
    window["earnings_timestamp"] = date
    market_events.append(window)

    valid_events.append(event["event_id"])

In [22]:
len(market_events), len(valid_events), len(failed_events)

(29574, 29574, 3788)

In [25]:
market_events[0]

Price,Close,High,Low,Open,Volume,relative_day,window,event_id,symbol,earnings_timestamp
Date,,,,,,,,,,
2020-10-13,101.290070,101.856960,100.848091,101.309290,915500,-29,pre,0,A,2020-11-23 16:30:00
2020-10-14,100.944153,102.491080,100.809638,101.357307,906700,-28,pre,0,A,2020-11-23 16:30:00
2020-10-15,101.193970,101.578301,99.810381,100.021763,723000,-27,pre,0,A,2020-11-23 16:30:00
2020-10-16,102.519928,103.327024,101.568714,101.799310,1039400,-26,pre,0,A,2020-11-23 16:30:00
2020-10-19,101.357300,103.403859,101.117094,102.596763,636000,-25,pre,0,A,2020-11-23 16:30:00
2020-10-20,101.472618,102.750516,101.424574,101.741648,771000,-24,pre,0,A,2020-11-23 16:30:00
2020-10-21,100.723183,102.510316,100.040999,101.770477,894000,-23,pre,0,A,2020-11-23 16:30:00
2020-10-22,102.587173,102.894637,100.992209,101.193982,1064700,-22,pre,0,A,2020-11-23 16:30:00
2020-10-23,102.010689,103.173285,101.338115,103.029167,833900,-21,pre,0,A,2020-11-23 16:30:00


In [ ]:
df_market_events = pd.concat(market_events, ignore_index=False)

In [32]:
df_market_events.head()

Price,Close,High,Low,Open,Volume,relative_day,window,event_id,symbol,earnings_timestamp
Date,,,,,,,,,,
2020-10-13,101.290070,101.856960,100.848091,101.309290,915500,-29,pre,0,A,2020-11-23 16:30:00
2020-10-14,100.944153,102.491080,100.809638,101.357307,906700,-28,pre,0,A,2020-11-23 16:30:00
2020-10-15,101.193970,101.578301,99.810381,100.021763,723000,-27,pre,0,A,2020-11-23 16:30:00
2020-10-16,102.519928,103.327024,101.568714,101.799310,1039400,-26,pre,0,A,2020-11-23 16:30:00
2020-10-19,101.357300,103.403859,101.117094,102.596763,636000,-25,pre,0,A,2020-11-23 16:30:00


In [29]:
# clean the original earnings transcripts dataframe to only include valid events
valid_events = set(valid_events)
df_clean = df[df["event_id"].isin(valid_events)].copy()
print(df_clean.head())
df_clean.shape, df.shape

  symbol  quarter  year                date  \
0      A        4  2020 2020-11-23 16:30:00   
1      A        3  2020 2020-08-18 16:30:00   
2      A        2  2020 2020-05-21 16:30:00   
3      A        1  2020 2020-02-18 16:30:00   
4      A        4  2021 2021-11-22 16:30:00   

                                             content  \
0  Operator: Good afternoon, and welcome to the A...   
1  Operator: Good afternoon, and welcome to the A...   
2  Operator: Good afternoon, and welcome to the A...   
3  Operator: Good afternoon and welcome to the Ag...   
4  Operator: Good afternoon and welcome to the Ag...   

                                  structured_content  \
0  [{'speaker': 'Operator', 'text': "Good afterno...   
1  [{'speaker': 'Operator', 'text': "Good afterno...   
2  [{'speaker': 'Operator', 'text': "Good afterno...   
3  [{'speaker': 'Operator', 'text': "Good afterno...   
4  [{'speaker': 'Operator', 'text': "Good afterno...   

                 company_name  company_id  

((29574, 9), (33362, 9))

In [30]:
df_clean.to_csv('earnings_transcripts_clean.csv', index=False)

In [33]:
df_market_events.to_csv('marketdata_to_events.csv', index=True)

In [34]:
# reduce the cleaned earnings transcripts dataframe to only the relevant columns
df_trscrpt = pd.read_csv('earnings_transcripts_clean.csv')
trscrpt_cols = ["event_id", "symbol", "date", "content"]
df_trscrpt_red = df_trscrpt[trscrpt_cols].copy()
df_trscrpt_red.head()

,event_id,symbol,date,content
0,0,A,2020-11-23 16:30:00,"Operator: Good afternoon, and welcome to the A..."
1,1,A,2020-08-18 16:30:00,"Operator: Good afternoon, and welcome to the A..."
2,2,A,2020-05-21 16:30:00,"Operator: Good afternoon, and welcome to the A..."
3,3,A,2020-02-18 16:30:00,Operator: Good afternoon and welcome to the Ag...
4,4,A,2021-11-22 16:30:00,Operator: Good afternoon and welcome to the Ag...


In [35]:
# sort earnings events chronologically
df_trscrpt_red = df_trscrpt_red.sort_values("date").reset_index(drop=True)
df_trscrpt_red.head()

,event_id,symbol,date,content
0,10404,AAPL,2005-10-13 14:45:00,TRANSCRIPT SPONSOR :\nApple's Q4 2005 Conferen...
1,32947,COP,2005-10-30 17:00:00,Operator: Ladies and gentlemen and welcome to ...
2,32026,GLW,2005-10-31 17:00:00,Operator: Good morning everyone. Welcome to t...
3,33031,XOM,2005-10-31 17:00:00,Operator: I would like to turn the call over t...
4,33161,XEL,2005-10-31 17:00:00,Operator: Good morning. My name is Dennis and...


In [36]:
df_trscrpt_red.to_csv('earnings_transcripts_sorted.csv', index=False)